In [ ]:
import pandas as pd
import numpy as np
init_seed = 42
np.random.seed(init_seed) 
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix,precision_recall_curve, roc_curve, auc, roc_auc_score
from sklearn import datasets, ensemble
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder, label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.inspection import permutation_importance
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier

from econml.dr import DRLearner

import xgboost as xgb
print(xgb.__version__)
from xgboost import XGBClassifier

2.1.3


In [ ]:
############import datafile################################################
file_path1 = 'data/Label_data_n_258.csv'
data = pd.read_csv(file_path1)#, engine='openpyxl'
data

,VEC_HEA,Omega_HEA,Hmix_HEA,Electroneg_HEA,DeltaSize_HEA,Smix_HEA,Al+Cr,Cr/Al,Refractory_sum,NiCoCrAlFe_sum,...,T(C),Pdflist,Entry,Oxidation Time,Oxidation Weight Gain (mg/cm2),cluster,kn,n,kn_err,n_err
0,5.90000,0.0,-2.520000,0.048000,0.042854,2.702740,90.000000,0.0,10.000000,90.000000,...,1100,1,1,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN
1,5.90000,0.0,-2.520000,0.048000,0.042854,2.702740,90.000000,0.0,10.000000,90.000000,...,1100,1,1,18.000000,4.120390,0.0,NaN,NaN,NaN,NaN
2,5.90000,0.0,-2.520000,0.048000,0.042854,2.702740,90.000000,0.0,10.000000,90.000000,...,1100,1,1,36.000000,5.824330,0.0,0.946893,2.002759,0.000000,0.000000
3,5.90000,0.0,-2.520000,0.048000,0.042854,2.702740,90.000000,0.0,10.000000,90.000000,...,1100,1,1,54.000000,7.010000,0.0,1.060853,2.075546,0.074484,0.038911
4,5.90000,0.0,-2.520000,0.048000,0.042854,2.702740,90.000000,0.0,10.000000,90.000000,...,1100,1,1,72.000000,7.527830,0.0,1.629717,2.333774,0.548810,0.178756
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32755,5.41736,0.0,-26.549443,0.096331,0.074681,8.008781,56.694406,0.0,28.347203,56.694406,...,1200,786,5057,30.038109,21.229542,1.0,508.994565,3.153408,0.000000,0.000000
32756,5.41736,0.0,-26.549443,0.096331,0.074681,8.008781,56.694406,0.0,28.347203,56.694406,...,1200,786,5057,45.057164,25.642753,1.0,121.763157,2.663350,81.926711,0.216908
32757,5.41736,0.0,-26.549443,0.096331,0.074681,8.008781,56.694406,0.0,28.347203,56.694406,...,1200,786,5057,60.076218,29.871108,0.0,53.820536,2.391688,30.976757,0.178783
32758,5.41736,0.0,-26.549443,0.096331,0.074681,8.008781,56.694406,0.0,28.347203,56.694406,...,1200,786,5057,75.095273,34.652044,0.0,25.260857,2.147013,14.179034,0.168374


In [3]:
###########check features################
fnum = 258
# features = data.columns[0:246].tolist()
features = data.columns[0:fnum].tolist()+['Oxidation Time']
print(f'features are:{features}')
##########check data#######################
# data[features]
print(f'data contain Nan: {data[features].isna().any().any()}')
print(f'data contain inf: {np.isinf(np.isinf(data.iloc[:,:252].values).any()).any()}')

features are:['VEC_HEA', 'Omega_HEA', 'Hmix_HEA', 'Electroneg_HEA', 'DeltaSize_HEA', 'Smix_HEA', 'Al+Cr', 'Cr/Al', 'Refractory_sum', 'NiCoCrAlFe_sum', 'ECVF', 'WAPBR_ZrO2', 'WAPBR_TiO2', 'WAPBR_Ta2O5', 'WAPBR_Nb2O5', 'WAPBR_WO3', 'WAPBR_MoO3', 'WAPBR_V2O5', 'WAPBR_Fe2O3', 'WAPBR_Al2O3', 'WAPBR_Co3O4', 'WAPBR_Cr2O3', 'WAPBR_NiO', 'WAPBR', 'WAOR_ZrO2', 'WAOR_TiO2', 'WAOR_Ta2O5', 'WAOR_Nb2O5', 'WAOR_WO3', 'WAOR_MoO3', 'WAOR_V2O5', 'WAOR_Fe2O3', 'WAOR_Al2O3', 'WAOR_Co3O4', 'WAOR_Cr2O3', 'WAOR_NiO', 'WAOR', 'WOMR_ZrO2', 'WOMR_TiO2', 'WOMR_Ta2O5', 'WOMR_Nb2O5', 'WOMR_WO3', 'WOMR_MoO3', 'WOMR_V2O5', 'WOMR_Fe2O3', 'WOMR_Al2O3', 'WOMR_Co3O4', 'WOMR_Cr2O3', 'WOMR_NiO', 'WOMR', 'WSD_Ni', 'WSD_Co', 'WSD_Cr', 'WSD_Al', 'WSD_Fe', 'WSD_Nb', 'WSD_Ta', 'WSD_Ti', 'WSD_Zr', 'WSD_Mo', 'WSD_W', 'WSD_V', 'WSD', 'WID_V', 'WID_W', 'WID_Mo', 'WID_Zr', 'WID_Ti', 'WID_Ta', 'WID_Nb', 'WID_Fe', 'WID_Al', 'WID_Cr', 'WID_Co', 'WID_Ni', 'WID', 'WODS_Zr', 'WODS_Ti', 'WODS_Ta', 'WODS_Nb', 'WODS_W', 'WODS_Mo', 'WODS_V',

In [4]:
#################################################################################
# Standardize the features
#################################################################################
# Columns to exclude
exclude_columns = ['cyc', 'dis', 'iso','N.1','Y.1']
##################Boolean to int################################# 
data[exclude_columns] = data[exclude_columns].astype(int)
########################################################################
# Remove elements in arrayA from arrayB
Numerical = list(set(features) - set(exclude_columns))
scaler = StandardScaler()
scaler.fit(data[Numerical])
###############################################################################################
data[Numerical] = scaler.transform(data[Numerical])
data

,VEC_HEA,Omega_HEA,Hmix_HEA,Electroneg_HEA,DeltaSize_HEA,Smix_HEA,Al+Cr,Cr/Al,Refractory_sum,NiCoCrAlFe_sum,...,T(C),Pdflist,Entry,Oxidation Time,Oxidation Weight Gain (mg/cm2),cluster,kn,n,kn_err,n_err
0,-0.485437,0.0,0.929520,-1.155079,-0.142336,-1.543374,4.116745,-0.102796,-0.488432,0.592892,...,0.761761,1,1,-0.259887,0.000000,0.0,NaN,NaN,NaN,NaN
1,-0.485437,0.0,0.929520,-1.155079,-0.142336,-1.543374,4.116745,-0.102796,-0.488432,0.592892,...,0.761761,1,1,-0.227408,4.120390,0.0,NaN,NaN,NaN,NaN
2,-0.485437,0.0,0.929520,-1.155079,-0.142336,-1.543374,4.116745,-0.102796,-0.488432,0.592892,...,0.761761,1,1,-0.194930,5.824330,0.0,0.946893,2.002759,0.000000,0.000000
3,-0.485437,0.0,0.929520,-1.155079,-0.142336,-1.543374,4.116745,-0.102796,-0.488432,0.592892,...,0.761761,1,1,-0.162451,7.010000,0.0,1.060853,2.075546,0.074484,0.038911
4,-0.485437,0.0,0.929520,-1.155079,-0.142336,-1.543374,4.116745,-0.102796,-0.488432,0.592892,...,0.761761,1,1,-0.129972,7.527830,0.0,1.629717,2.333774,0.548810,0.178756
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32755,-0.754256,0.0,-1.278191,-0.310916,0.860840,-0.057465,2.064229,-0.102796,0.039716,-0.295791,...,1.318022,786,5057,-0.205687,21.229542,1.0,508.994565,3.153408,0.000000,0.000000
32756,-0.754256,0.0,-1.278191,-0.310916,0.860840,-0.057465,2.064229,-0.102796,0.039716,-0.295791,...,1.318022,786,5057,-0.178587,25.642753,1.0,121.763157,2.663350,81.926711,0.216908
32757,-0.754256,0.0,-1.278191,-0.310916,0.860840,-0.057465,2.064229,-0.102796,0.039716,-0.295791,...,1.318022,786,5057,-0.151487,29.871108,0.0,53.820536,2.391688,30.976757,0.178783
32758,-0.754256,0.0,-1.278191,-0.310916,0.860840,-0.057465,2.064229,-0.102796,0.039716,-0.295791,...,1.318022,786,5057,-0.124387,34.652044,0.0,25.260857,2.147013,14.179034,0.168374


In [5]:
#################################SMOTE#######################################
tem = data.copy()
tem = tem[~tem['cluster'].isna()]
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)
X_smote, T_smote = sm.fit_resample(tem[features], tem['cluster'])
X_smote

,VEC_HEA,Omega_HEA,Hmix_HEA,Electroneg_HEA,DeltaSize_HEA,Smix_HEA,Al+Cr,Cr/Al,Refractory_sum,NiCoCrAlFe_sum,...,Ag,Au,pO2(mmHg),cyc,dis,iso,N.1,Y.1,T(C),Oxidation Time
0,-0.485437,0.0,0.929520,-1.155079,-0.142336,-1.543374,4.116745,-0.102796,-0.488432,0.592892,...,0.0,-0.018595,-0.115233,1,0,0,1,0,0.761761,-0.259887
1,-0.485437,0.0,0.929520,-1.155079,-0.142336,-1.543374,4.116745,-0.102796,-0.488432,0.592892,...,0.0,-0.018595,-0.115233,1,0,0,1,0,0.761761,-0.227408
2,-0.485437,0.0,0.929520,-1.155079,-0.142336,-1.543374,4.116745,-0.102796,-0.488432,0.592892,...,0.0,-0.018595,-0.115233,1,0,0,1,0,0.761761,-0.194930
3,-0.485437,0.0,0.929520,-1.155079,-0.142336,-1.543374,4.116745,-0.102796,-0.488432,0.592892,...,0.0,-0.018595,-0.115233,1,0,0,1,0,0.761761,-0.162451
4,-0.485437,0.0,0.929520,-1.155079,-0.142336,-1.543374,4.116745,-0.102796,-0.488432,0.592892,...,0.0,-0.018595,-0.115233,1,0,0,1,0,0.761761,-0.129972
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53293,1.209938,0.0,0.653920,0.063198,-1.024149,-0.879811,0.165890,-0.102796,-0.776296,0.843892,...,0.0,-0.018595,-0.115233,0,1,0,0,1,0.205501,0.982847
53294,0.917600,0.0,-0.295237,0.433166,0.294578,0.340846,-0.181108,-0.097693,-0.551450,0.651305,...,0.0,-0.018595,-0.115233,0,1,0,1,0,1.318022,-0.164665
53295,0.998945,0.0,0.484098,0.280599,-0.281310,0.065052,0.243454,0.041810,-0.616189,0.674814,...,0.0,-0.018595,-0.115233,0,1,0,0,1,0.761761,0.532275
53296,0.795608,0.0,1.009415,-0.459847,-0.329197,0.336184,-0.320382,-0.102796,-0.776296,0.406113,...,0.0,-0.018595,-0.115233,0,0,1,0,1,0.205501,-0.135648


In [ ]:
############import datafile################################################
corr_matrix = pd.read_csv("data/pearson_correlation.csv", index_col=0)#, engine='openpyxl'
shap_df = pd.read_csv("data/total SHAP importance_RF100n211.csv")#, engine='openpyxl'
rankindex = 'avg SHAP Importance' #avg SHAP Importance   Geometric Mean SHAP Importance
shap_df = shap_df.sort_values(by=rankindex, ascending=False)
shap_df

,Unnamed: 0,Feature,avg SHAP Importance,Geometric Mean SHAP Importance,Rank SHAP Importance
258,258,Oxidation Time,0.065442,0.063729,257.0
256,256,Y.1,0.035239,0.017271,209.0
252,252,cyc,0.031084,0.017827,223.0
108,108,WVOC_MoO3,0.023279,0.016221,233.5
254,254,iso,0.018096,0.013191,222.0
...,...,...,...,...,...
102,102,WVOC_NiO,0.000000,0.000000,8.0
113,113,WVOC_ZrO2,0.000000,0.000000,16.5
112,112,WVOC_TiO2,0.000000,0.000000,4.0
105,105,WVOC_Al2O3,0.000000,0.000000,8.0


In [7]:
####################Identify Highly Correlated Features (Threshold = 0.85)######################################
corr_matrix_abs = corr_matrix.abs()

# Identify highly correlated feature pairs (|correlation| > 0.85)
high_corr_pairs = set()
for i in range(len(corr_matrix_abs.columns)):
    for j in range(i + 1, len(corr_matrix_abs.columns)):
        if corr_matrix_abs.iloc[i, j] > 0.85:
            high_corr_pairs.add((corr_matrix_abs.columns[i], corr_matrix_abs.columns[j],corr_matrix_abs.iloc[i, j]))

print("Highly correlated pairs:", high_corr_pairs)
# Convert to DataFrame
high_corr_df = pd.DataFrame(high_corr_pairs, columns=["Feature1", "Feature2", "Correlation"])

Highly correlated pairs: {('WODS_Cr', 'WOSS_Cr', 0.8669642472097573), ('WOSS_Cr', 'OGf_CrO3', 0.8756626222532107), ('WSD_Ni', 'WOSS_Ni', 0.8615579537710494), ('WSD_Al', 'Al', 0.9492877225436652), ('WOSS_Ni', 'Ni', 0.8597287601353802), ('WAPBR_Cr2O3', 'WOMS_Cr2O3', 0.9582226652237584), ('WAOR', 'MaxG', 0.9324292579798134), ('WODS_Ni', 'WAPO', 0.8659357243371589), ('WAPBR_Al2O3', 'WSD_Al', 0.9499222710076296), ('WOMR_Al2O3', 'WODS_Al', 0.9478572341282866), ('VEC_HEA', 'WSS_Al2O3', 0.906414728211196), ('WSD', 'OGf_CrO3', 0.9583402036324687), ('WOMR_Cr2O3', 'WOSS_Cr', 0.8956243534000071), ('WSD', 'OGf_Cr2O3', 0.9592523740129172), ('WAPBR_NiO', 'WOMR_NiO', 0.8947580567697437), ('WOSS_Cr', 'WVOC_Cr2O3', 0.8568651721236834), ('WAPBR_Cr2O3', 'OGf_Cr2O3', 0.9663295662878632), ('WSD_Ni', 'WOMS_NiO', 0.8852275514634745), ('WOMS_NiO', 'OGf_NiO', 0.8935523588137991), ('WODS_Ni', 'WAPO_NiO', 0.8931639032539066), ('WODS_Al', 'OGf_Al2O3', 0.94674962659138), ('OGf_NiO', 'Ni', 0.8891184788807623), ('WOM

In [8]:
# Remove the less important feature from each correlated pair
features_to_remove = []
f1shap = []
f2shap = []
for f1, f2, _ in high_corr_pairs:
    f1shap.append(shap_df.loc[shap_df["Feature"] == f1, rankindex].iloc[0])
    f2shap.append(shap_df.loc[shap_df["Feature"] == f2, rankindex].iloc[0])
    if shap_df.loc[shap_df["Feature"] == f1, rankindex].iloc[0] <= shap_df.loc[shap_df["Feature"] == f2, rankindex].iloc[0]:
        features_to_remove.append(f1)
    else:
        features_to_remove.append(f2)
high_corr_df['f1 Weighted SHAP Importance']=f1shap
high_corr_df['f2 Weighted SHAP Importance']=f2shap
high_corr_df['Removed']=features_to_remove
##################################################################
#################Summary Output##################################################################
print("\n🎯 Summary of Process:")
print(f"- Features removed: {set(features_to_remove)}")
print(f"- Remaining features: {list(data.columns)}")


🎯 Summary of Process:
- Features removed: {'WOMR_Al2O3', 'WODS_Al', 'WSS_Al2O3', 'WVOC_Cr2O3', 'WOMR_Cr2O3', 'OGf_Cr2O3', 'Refractory_sum', 'WSD_Cr', 'OGf_CrO3', 'WOSS_Ni', 'WAPO', 'Al', 'WOSS_Cr', 'WODS_Cr', 'WOMS_NiO', 'VEC_HEA', 'WAPBR_Al2O3', 'WSD_Al', 'Cr', 'WAPBR_Cr2O3', 'WOMR_NiO', 'OGf_NiO', 'MaxG', 'WSS_Cr2O3', 'WAPBR_NiO', 'WSD_Ni', 'ECVF', 'OGf_Al2O3', 'WAPO_NiO', 'WOMS_Cr2O3', 'Ni'}
- Remaining features: ['VEC_HEA', 'Omega_HEA', 'Hmix_HEA', 'Electroneg_HEA', 'DeltaSize_HEA', 'Smix_HEA', 'Al+Cr', 'Cr/Al', 'Refractory_sum', 'NiCoCrAlFe_sum', 'ECVF', 'WAPBR_ZrO2', 'WAPBR_TiO2', 'WAPBR_Ta2O5', 'WAPBR_Nb2O5', 'WAPBR_WO3', 'WAPBR_MoO3', 'WAPBR_V2O5', 'WAPBR_Fe2O3', 'WAPBR_Al2O3', 'WAPBR_Co3O4', 'WAPBR_Cr2O3', 'WAPBR_NiO', 'WAPBR', 'WAOR_ZrO2', 'WAOR_TiO2', 'WAOR_Ta2O5', 'WAOR_Nb2O5', 'WAOR_WO3', 'WAOR_MoO3', 'WAOR_V2O5', 'WAOR_Fe2O3', 'WAOR_Al2O3', 'WAOR_Co3O4', 'WAOR_Cr2O3', 'WAOR_NiO', 'WAOR', 'WOMR_ZrO2', 'WOMR_TiO2', 'WOMR_Ta2O5', 'WOMR_Nb2O5', 'WOMR_WO3', 'WOMR_MoO3', 'WOMR

In [ ]:
# check
shap_df_filtered = shap_df[~shap_df["Feature"].isin(features_to_remove)]
shap_df_filtered[shap_df_filtered['Feature'].str.contains('_Nb', na=False)]

,Unnamed: 0,Feature,avg SHAP Importance,Geometric Mean SHAP Importance,Rank SHAP Importance
55,55,WSD_Nb,0.004771,0.004082,191.0
27,27,WAOR_Nb2O5,0.004421,0.003905,180.0
181,181,OGf_Nb2O5,0.004112,0.003157,168.5
92,92,WOSS_Nb,0.003960,0.003390,168.0
118,118,WOMS_Nb2O5,0.003921,0.003802,168.0
79,79,WODS_Nb,0.003917,0.003678,166.5
40,40,WOMR_Nb2O5,0.003571,0.000000,136.5
182,182,OGf_NbO2,0.003469,0.001521,147.0
69,69,WID_Nb,0.003371,0.003295,157.0
14,14,WAPBR_Nb2O5,0.003330,0.002001,144.0


In [10]:
from sklearn.linear_model import LogisticRegression, RidgeCV,LogisticRegression
from sklearn.linear_model import LassoCV, RidgeCV
from numpy.linalg import cond
import xgboost as xgb
from sklearn.tree import DecisionTreeRegressor

In [ ]:
RANDOM_STATE = 24
def causal_function(T_smote,X_smote,raw_data,features,treatment):
    data_final = raw_data.copy()
# Loop over each treatment type (1, 2, and 3)
    for treat in range(treatment):
        # Prepare data: X (features), y (outcome), and T_binary (binary indicator)
        X = raw_data[features].values#
        y = raw_data['Oxidation Weight Gain (mg/cm2)'].values
        # Create a binary treatment indicator: 1 if treatment equals 'treat', 0 otherwise.
        data_final[f'T_{treat}'] = (raw_data['cluster'] == treat).astype(int)
        T_binary = data_final[f'T_{treat}'].values.astype(int)
       #######define clf model, fit first cause the databse is balenced ##################
        X_smote_copy = X_smote[features].values
        T_smote_copy = (T_smote== treat).astype(int)
        model_clf = xgb.XGBClassifier(random_state=RANDOM_STATE,
            n_estimators=40,max_depth=10,learning_rate=0.5,subsample=0.9,min_child_weight=3)
        model_clf.fit(X_smote_copy, T_smote_copy)  
        #######define reg model##################
        # Specify machine learning models for the two nuisance functions.
        model_reg = ensemble.RandomForestRegressor(random_state=RANDOM_STATE,n_estimators=100, max_depth=None,min_samples_split=2,min_samples_leaf=1,max_features=1)#
        #######define final model##################
        model_final = RidgeCV(alphas=np.logspace(-3, 3, 7))
        # Initialize DRLearner (which expects a binary treatment)
        dr_learner = DRLearner(model_regression=model_reg,
                               model_propensity=model_clf,
                               model_final=model_final,
                               random_state=RANDOM_STATE,
                               cv=5)  # 
        # Fit the DRLearner on the binary treatment indicator.
        dr_learner.fit(y, T_binary, X=X)
        # Estimate the Conditional Average Treatment Effect (CATE) for each individual
        cate = dr_learner.effect(X)
        data_final[f'CATE_T_{treat}'] = cate
        # The average effect for treatment 't' (versus not receiving t) is:
        ate = np.mean(cate)
      
        print(f"Estimated ATE for treatment {treat} (vs. not {treat}): {ate:.3f}")
    return data_final

In [173]:
dfs = {}
##########################initial no feature engineering ###############################
features = data.columns[0:fnum].tolist()+['Oxidation Time']
print(f"The remaining features is: {len(features)}") 

dfs[f'CSVI threadhold:Initial'] = causal_function(T_smote,X_smote,data,features,3)

The remaining features is: 259
Estimated ATE for treatment 0 (vs. not 0): 1.165
Estimated ATE for treatment 1 (vs. not 1): -0.654
Estimated ATE for treatment 2 (vs. not 2): -6.019


In [ ]:
##########################just remove correlate feature keep all CSVI features ###############################
##############################remove hightle corelate features####################################################################################################################################
shap_df_filtered = shap_df[~shap_df["Feature"].isin(features_to_remove)]
##########################################################################################
CSVI_limt = [0,1e-6,5e-6,1e-5,5e-5,1e-4,5e-4,1e-3,5e-3,0.01,0.02,0.03]
for threshold in CSVI_limt:
    print("#############################")
    print(f"The CSVI threadhold is: {threshold}")
##############################remove low CSVI features######################################################################################################
    shap_df_final = shap_df_filtered[shap_df_filtered[rankindex]>=threshold]  
    print("#############################")
    features = shap_df_final['Feature'].tolist()
    print(f"The remaining high CSVI features is: {len(features)}") 
    ####################################################################################################
    plotvalue = len(features)
    plt.figure(figsize=(10, 5))
    plt.plot(shap_df_final["Feature"][:plotvalue], shap_df_final[rankindex][:plotvalue], marker="o", linestyle="-", color="b")
    plt.ylabel("CSVI")
    plt.xticks(rotation=45)  # Rotate x labels if needed
    plt.axhline(y=threshold, color='r', linestyle='--')#, label="y = 10"
    # Show the plot
    plt.show()
    ##########################################################################################
    dfs[f'CSVI threadhold:{threshold}'] = causal_function(T_smote,X_smote,data,features,3)

In [92]:
from sklearn.ensemble import GradientBoostingRegressor
import xgboost as xgb
from sklearn.metrics import make_scorer,r2_score,root_mean_squared_error, mean_squared_error,mean_absolute_error,classification_report, confusion_matrix,precision_recall_curve, roc_curve, auc, roc_auc_score
from sklearn.model_selection import GridSearchCV,KFold, cross_val_score, train_test_split,cross_validate
RANDOM_STATE = None

In [93]:
reg_results = {}
clf_results = {}
features_num = []

In [ ]:
##########################initial no feature engineering ###############################
features = data.columns[0:fnum].tolist()+['Oxidation Time']
features_num.append(len(features))
print(f"The remaining features is: {len(features)}") 
######reg#########reg############reg##############reg#####reg##############reg#########reg############reg##############reg#####reg###########
# Specify machine learning models for the two nuisance functions.
model_reg = ensemble.RandomForestRegressor(random_state=RANDOM_STATE,n_estimators=100, max_depth=None,min_samples_split=2,min_samples_leaf=1,max_features=1)#
######################################################################################################
# Define the scoring metrics
scoring = {
    'RMSE':make_scorer(root_mean_squared_error),
    'MAE': make_scorer(mean_absolute_error),
    'MSE': make_scorer(mean_squared_error),
    'R2': make_scorer(r2_score)
}
# Perform five-fold cross-validation on the training set
kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(model_reg, data[features], data['Oxidation Weight Gain (mg/cm2)'], cv=kfold, scoring=scoring)
reg_results[f'CSVI threadhold:Initial'] = cv_results
# # Extract and print the cross-validation results
print("Train CV RMSE Score:", cv_results['test_RMSE'].mean(),'±',cv_results['test_RMSE'].std())
print("Train CV MAE Score:", cv_results['test_MAE'].mean(),'±',cv_results['test_MAE'].std())
print("Train CV MSE Score:", cv_results['test_MSE'].mean(),'±',cv_results['test_MSE'].std())
print("Train CV R2 Score:", cv_results['test_R2'].mean(),'±',cv_results['test_R2'].std())
#######clf############clf#################clf#########################clf#####################clf#############clf###########clf############clf############    
params = {
    'n_estimators': 40,
    'max_depth': 10,
    'learning_rate': 0.5,
    'subsample': 0.9,
    'min_child_weight': 3,}
XGBoost_clf = xgb.XGBClassifier(objective='multi:softmax', num_class=3, eval_metric='mlogloss',**params,random_state=RANDOM_STATE)
# 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True,random_state=RANDOM_STATE)
# Store reports
all_reports = []
# Cross-validation loop
for train_idx, test_idx in kf.split(X_smote, T_smote):
    X_train, X_test = X_smote[features].iloc[train_idx], X_smote[features].iloc[test_idx]
    y_train, y_test = T_smote.iloc[train_idx], T_smote.iloc[test_idx]

    XGBoost_clf.fit(X_train, y_train)
    y_pred = XGBoost_clf.predict(X_test)

    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    all_reports.append(report)
# Convert to DataFrames and average
# We'll exclude 'accuracy' for now since it's a scalar (not per-class)
report_dfs = [pd.DataFrame(r).T for r in all_reports]

# Align all columns first (in case any class is missing in a fold)
for df in report_dfs:
    for metric in ['precision', 'recall', 'f1-score', 'support']:
        if metric not in df.columns:
            df[metric] = 0
# Stack and average
combined_df = pd.concat(report_dfs).groupby(level=0).mean()
# Add average accuracy separately
accuracy_scores = [r['accuracy'] for r in all_reports]
combined_df.loc['accuracy'] = [np.mean(accuracy_scores), 0, 0, 0]  # pad other columns
# Final averaged classification report
print("\n=== Averaged Classification Report ===")
print(combined_df)
clf_results[f'CSVI threadhold:Initial'] = combined_df

The remaining features is: 259
Train CV RMSE Score: 16.8041016419161 ± 2.446077221809423
Train CV MAE Score: 3.8026638162074393 ± 0.12222266167057663
Train CV MSE Score: 288.36112576690226 ± 84.81128181897392
Train CV R2 Score: 0.8795769423682364 ± 0.027250941904083847

=== Averaged Classification Report ===
              precision    recall  f1-score  support
0.0            0.866073  0.885390  0.875590   3553.2
1.0            0.879211  0.852565  0.865644   3553.2
2.0            0.988212  0.996115  0.992148   3553.2
accuracy       0.911329  0.000000  0.000000      0.0
macro avg      0.911165  0.911357  0.911127  10659.6
weighted avg   0.911182  0.911329  0.911121  10659.6


In [ ]:
##########################just remove correlate feature keep all CSVI features ###############################
##############################remove hightle corelate features####################################################################################################################################
shap_df_filtered = shap_df[~shap_df["Feature"].isin(features_to_remove)]
##########################################################################################
CSVI_limt = [0,1e-6,5e-6,1e-5,5e-5,1e-4,5e-4,1e-3,5e-3,0.01,0.02,0.03]
for threshold in CSVI_limt:
    print("#############################")
    print(f"The CSVI threadhold is: {threshold}")
##############################remove low CSVI features######################################################################################################
    shap_df_final = shap_df_filtered[shap_df_filtered[rankindex]>=threshold]  
    print("#############################")
    features = shap_df_final['Feature'].tolist()
    print(f"The remaining high CSVI features is: {len(features)}") 
    features_num.append(len(features))
######reg#########reg############reg##############reg#####reg##############reg#########reg############reg##############reg#####reg###########
    model_reg = ensemble.RandomForestRegressor(random_state=RANDOM_STATE,n_estimators=100, max_depth=None,min_samples_split=2,min_samples_leaf=1,max_features=1)#
    ######################################################################################################
    # Define the scoring metrics
    scoring = {
        'RMSE':make_scorer(root_mean_squared_error),
        'MAE': make_scorer(mean_absolute_error),
        'MSE': make_scorer(mean_squared_error),
        'R2': make_scorer(r2_score)
    }
    # Perform five-fold cross-validation on the training set
    kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_results = cross_validate(model_reg, data[features], data['Oxidation Weight Gain (mg/cm2)'], cv=kfold, scoring=scoring)
    reg_results[f'CSVI threadhold:{threshold}'] = cv_results
    # # Extract and print the cross-validation results
    print("Train CV RMSE Score:", cv_results['test_RMSE'].mean(),'±',cv_results['test_RMSE'].std())
    print("Train CV MAE Score:", cv_results['test_MAE'].mean(),'±',cv_results['test_MAE'].std())
    print("Train CV MSE Score:", cv_results['test_MSE'].mean(),'±',cv_results['test_MSE'].std())
    print("Train CV R2 Score:", cv_results['test_R2'].mean(),'±',cv_results['test_R2'].std())
#######clf############clf#################clf#########################clf#####################clf#############clf###########clf############clf############    
    params = {
        'n_estimators': 40,
        'max_depth': 10,
        'learning_rate': 0.5,
        'subsample': 0.9,
        'min_child_weight': 3,}
    XGBoost_clf = xgb.XGBClassifier(objective='multi:softmax', num_class=3, eval_metric='mlogloss',**params,random_state=RANDOM_STATE)
    # 5-Fold Cross-Validation
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    # Store reports
    all_reports = []
    # Cross-validation loop
    for train_idx, test_idx in kf.split(X_smote, T_smote):
        X_train, X_test = X_smote[features].iloc[train_idx], X_smote[features].iloc[test_idx]
        y_train, y_test = T_smote.iloc[train_idx], T_smote.iloc[test_idx]
    
        XGBoost_clf.fit(X_train, y_train)
        y_pred = XGBoost_clf.predict(X_test)
    
        report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
        all_reports.append(report)
    # Convert to DataFrames and average
    # We'll exclude 'accuracy' for now since it's a scalar (not per-class)
    report_dfs = [pd.DataFrame(r).T for r in all_reports]
    
    # Align all columns first (in case any class is missing in a fold)
    for df in report_dfs:
        for metric in ['precision', 'recall', 'f1-score', 'support']:
            if metric not in df.columns:
                df[metric] = 0
    # Stack and average
    combined_df = pd.concat(report_dfs).groupby(level=0).mean()
    # Add average accuracy separately
    accuracy_scores = [r['accuracy'] for r in all_reports]
    combined_df.loc['accuracy'] = [np.mean(accuracy_scores), 0, 0, 0]  # pad other columns
    # Final averaged classification report
    print("\n=== Averaged Classification Report ===")
    print(combined_df)
    clf_results[f'CSVI threadhold:{threshold}'] = combined_df


#############################
The CSVI threadhold is: 0
#############################
The remaining high CSVI features is: 228
Train CV RMSE Score: 16.71008227137603 ± 2.263472893643365
Train CV MAE Score: 3.7932054413127836 ± 0.2200137823121055
Train CV MSE Score: 284.3501590564138 ± 76.72274266302551
Train CV R2 Score: 0.8810801282454402 ± 0.030908586587514227

=== Averaged Classification Report ===
              precision    recall  f1-score  support
0.0            0.870754  0.889443  0.879965   3553.2
1.0            0.882806  0.857007  0.869663   3553.2
2.0            0.987595  0.995380  0.991469   3553.2
accuracy       0.913899  0.000000  0.000000      0.0
macro avg      0.913718  0.913943  0.913699  10659.6
weighted avg   0.913777  0.913899  0.913705  10659.6
#############################
The CSVI threadhold is: 1e-06
#############################
The remaining high CSVI features is: 214
Train CV RMSE Score: 18.516374088318127 ± 4.349329507079305
Train CV MAE Score: 3.88005108527

In [11]:
import pickle

# with open('Step5/dataframes_regresult_RF100n211_navg.pkl', 'wb') as f:
#     pickle.dump(reg_results, f)
# with open('Step5/dataframes_clfresult_XG_RF100n211_navg.pkl', 'wb') as f:
#     pickle.dump(clf_results, f)
# with open('Step5/features_nu_RF100n211_navg.pkl', 'wb') as f:
#     pickle.dump(features_num, f)
with open('Step5/dataframes_regresult_RF100n211_24avg.pkl', 'rb') as f:
    reg_results = pickle.load(f)
with open('Step5/dataframes_clfresult_XG_RF100n211_24avg.pkl', 'rb') as f:
    clf_results = pickle.load(f)
with open('Step5/features_nu_RF100n211_24avg.pkl', 'rb') as f:
    features_num = pickle.load(f)




In [ ]:
plotname = [0,1e-6,5e-6,1e-5,5e-5,1e-4,5e-4,1e-3,5e-3,0.01,0.02,0.03]
RMSE=[]
MAE=[]
MSE=[]
for threshold in plotname:
    RMSE.append(reg_results[f'CSVI threadhold:{threshold}']['test_RMSE'].mean())
    MAE.append(reg_results[f'CSVI threadhold:{threshold}']['test_MAE'].mean())
    MSE.append(reg_results[f'CSVI threadhold:{threshold}']['test_MSE'].mean())

fig, axes = plt.subplots(1, 3, figsize=(12, 5))
axes[0].plot(plotname, RMSE, linestyle='-', marker='o', color='b', label='RMSE')
axes[0].set_xlabel('CSVI threshold')
axes[0].set_ylabel('Performance')
# plt.grid(True)
axes[0].legend()
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)

axes[1].plot(plotname, MAE, linestyle='-', marker='*', color='r', label='MAE')
axes[1].set_xlabel('CSVI threshold')
axes[1].set_ylabel('Performance')
# plt.grid(True)
axes[1].legend()
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)

axes[2].plot(plotname, MSE, linestyle='-', marker='^', color='g', label='MSE')
axes[2].set_xlabel('CSVI threshold')
axes[2].set_ylabel('Performance')
# plt.grid(True)
axes[2].legend()
axes[2].set_xticklabels(axes[1].get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

# Minimum value and index in A
min_A = np.min(RMSE)
idx_A = np.argmin(RMSE)

# Minimum value and index in B
min_B = np.min(MSE)
idx_B = np.argmin(MSE)

# Minimum value and index in C
min_C = np.min(MAE)
idx_C = np.argmin(MAE)

# Print results
print(f"RMSE: min = {min_A},CSVI threshold = {plotname[idx_A]}, number of features = {features_num[idx_A]}")
print(f"MSE: min = {min_B},CSVI threshold = {plotname[idx_B]}, number of features = {features_num[idx_B]}")
print(f"MASE: min = {min_C},CSVI threshold = {plotname[idx_C]}, number of features = {features_num[idx_C]}")

In [4]:
fig, ax1 = plt.subplots()

# Plot on left y-axis
line1, = ax1.plot(plotname, RMSE, linestyle='-', marker='o', color='b', label='RMSE')
line2, = ax1.plot(plotname, MAE, linestyle='-', marker='*', color='r', label='MAE')
ax1.set_ylabel('Performance (RMSE & MAE)')

# Create right y-axis
ax2 = ax1.twinx()
line3, = ax2.plot(plotname, MSE, linestyle='-', marker='^', color='g', label='MSE')
ax2.set_ylabel('MSE')

# Combine legends
lines = [line1, line2, line3]
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, loc='upper left')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45)
ax1.set_xlabel('CSVI threshold')
plt.tight_layout()
plt.savefig(f"plot/Step5_causal_reg.png", dpi=800)  # You can change the filename and dpi
plt.show()

NameError: name 'plt' is not defined

In [5]:
T0_precision=[]
T0_recall=[]
T0_f1=[]
T1_precision=[]
T1_recall=[]
T1_f1=[]
T2_precision=[]
T2_recall=[]
T2_f1=[]

for threshold in plotname:
    T0_precision.append(clf_results[f'CSVI threadhold:{threshold}']['precision'].iloc[0])
    T0_recall.append(clf_results[f'CSVI threadhold:{threshold}']['recall'].iloc[0])
    T0_f1.append(clf_results[f'CSVI threadhold:{threshold}']['f1-score'].iloc[0])
    T1_precision.append(clf_results[f'CSVI threadhold:{threshold}']['precision'].iloc[1])
    T1_recall.append(clf_results[f'CSVI threadhold:{threshold}']['recall'].iloc[1])
    T1_f1.append(clf_results[f'CSVI threadhold:{threshold}']['f1-score'].iloc[1])
    T2_precision.append(clf_results[f'CSVI threadhold:{threshold}']['precision'].iloc[2])
    T2_recall.append(clf_results[f'CSVI threadhold:{threshold}']['recall'].iloc[2])
    T2_f1.append(clf_results[f'CSVI threadhold:{threshold}']['f1-score'].iloc[2])
    
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
axes[0].plot(plotname, T0_precision, linestyle='-', marker='^', color='g', label='precision')
axes[0].plot(plotname, T0_recall, linestyle='-', marker='o', color='b', label='recall')
axes[0].plot(plotname, T0_f1, linestyle='-', marker='*', color='r', label='f1')
axes[0].set_xlabel('CSVI threshold')
axes[0].set_ylabel('Performance')
axes[0].set_title('PL')
axes[0].legend()
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)

axes[1].plot(plotname, T1_precision, linestyle='-', marker='^', color='g', label='precision')
axes[1].plot(plotname, T1_recall, linestyle='-', marker='o', color='b', label='recall')
axes[1].plot(plotname, T1_f1, linestyle='-', marker='*', color='r', label='f1')
axes[1].set_xlabel('CSVI threshold')
axes[1].set_ylabel('Performance')
axes[1].set_title('NPL')
axes[1].legend()
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)

axes[2].plot(plotname, T2_precision, linestyle='-', marker='^', color='g', label='precision')
axes[2].plot(plotname, T2_recall, linestyle='-', marker='o', color='b', label='recall')
axes[2].plot(plotname, T2_f1, linestyle='-', marker='*', color='r', label='f1')
axes[2].set_xlabel('CSVI threshold')
axes[2].set_ylabel('Performance')
axes[2].set_title('WL')
axes[2].legend()
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=45)
plt.tight_layout()
plt.savefig(f"plot/Step5_causal_clf.png", dpi=800)  # You can change the filename and dpi
plt.show()

# Minimum value and index in A
min_A = np.max(T0_precision)
idx_A = np.argmax(T0_precision)
# Minimum value and index in B
min_B = np.max(T0_recall)
idx_B = np.argmax(T0_recall)
# Minimum value and index in C
min_C = np.max(T0_f1)
idx_C = np.argmax(T0_f1)
# Print results
print(f"T0_precision: max = {min_A},CSVI threshold = {plotname[idx_A]}, number of features = {features_num[idx_A]}")
print(f"T0_recall: max = {min_B},CSVI threshold = {plotname[idx_B]}, number of features = {features_num[idx_B]}")
print(f"T0_f1: max = {min_C},CSVI threshold = {plotname[idx_C]}, number of features = {features_num[idx_C]}")

# Minimum value and index in A
min_A = np.max(T1_precision)
idx_A = np.argmax(T1_precision)
# Minimum value and index in B
min_B = np.max(T1_recall)
idx_B = np.argmax(T1_recall)
# Minimum value and index in C
min_C = np.max(T1_f1)
idx_C = np.argmax(T1_f1)
# Print results
print(f"T1_precision: max = {min_A},CSVI threshold = {plotname[idx_A]}, number of features = {features_num[idx_A]}")
print(f"T1_recall: max = {min_B},CSVI threshold = {plotname[idx_B]}, number of features = {features_num[idx_B]}")
print(f"T1_f1: max = {min_C},CSVI threshold = {plotname[idx_C]}, number of features = {features_num[idx_C]}")

# Minimum value and index in A
min_A = np.max(T2_precision)
idx_A = np.argmax(T2_precision)
# Minimum value and index in B
min_B = np.max(T2_recall)
idx_B = np.argmax(T2_recall)
# Minimum value and index in C
min_C = np.max(T2_f1)
idx_C = np.argmax(T2_f1)
# Print results
print(f"T2_precision: max = {min_A},CSVI threshold = {plotname[idx_A]}, number of features = {features_num[idx_A]}")
print(f"T2_recall: max = {min_B},CSVI threshold = {plotname[idx_B]}, number of features = {features_num[idx_B]}")
print(f"T2_f1: max = {min_C},CSVI threshold = {plotname[idx_C]}, number of features = {features_num[idx_C]}")



NameError: name 'plotname' is not defined